# Cleaning the products dataset

In [ ]:
import pandas as pd

In [ ]:
#Read in the data
path = '../data/data_raw/products.csv'
products_orig = pd.read_csv(path)

In [ ]:
products = products_orig.copy()

### Dropping duplicate rows

In [ ]:
print(f'{len(products)} rows in products dataset')
print(f'{products.duplicated().sum()} duplicated rows')

#remove duplicated rows
products = products.drop_duplicates()
print(f'{len(products)} rows after dropping duplicates')

19326 rows in products dataset
8746 duplicated rows
10580 rows after dropping duplicates


### Check for duplicated 'sku'
'sku's should be unique.


In [ ]:
# Are there any duplicate skus?
print(f'\n{products.duplicated('sku').sum()} duplicated sku values')

# investigate:
sku_dupe = products.loc[products.duplicated('sku'), 'sku']
products.loc[products['sku'].isin(sku_dupe)]


1 duplicated sku values


,sku,name,desc,price,promo_price,in_stock,type
7992,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,1729,1305.59,0,1282
8000,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,NaN,1305.59,0,1282


The duplicated sku is identical to another row, but is missing the price - it can be removed.

In [ ]:
print(f'{len(products)} rows before dropping duplicate sku')
products = products.drop_duplicates('sku')
print(f'{len(products)} rows after dropping duplicate sku')

10580 rows before dropping duplicate sku
10579 rows after dropping duplicate sku


### Initial data exploration

In [ ]:
products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10579 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          10579 non-null  object
 1   name         10579 non-null  object
 2   desc         10572 non-null  object
 3   price        10534 non-null  object
 4   promo_price  10579 non-null  object
 5   in_stock     10579 non-null  int64 
 6   type         10529 non-null  object
dtypes: int64(1), object(6)
memory usage: 661.2+ KB


In [ ]:
products.isna().sum()

,0
sku,0
name,0
desc,7
price,45
promo_price,0
in_stock,0
type,50


'price' and 'promo_price' are strings, and there are some missing values in desc, price, and type.

### Investigate why price and promo_price are strings and not numeric

In [ ]:
products.sample(10)

,sku,name,desc,price,promo_price,in_stock,type
11661,KEN0236,Kensington MP230L black mouse accuracy,Precision Wireless Mouse for Mac and PC.,59.99,529.895,0,1387
19045,BOS0035-A,Open - Bose SoundTouch 20 Series III Speaker B...,Wireless music system compatible with iOS and ...,399,3.291.582,0,5398
13498,WAC0202,Wacom Bamboo Stylus Solo 4 Red,with standard digital pen tip and triangular d...,19.9,129.954,1,1229
12599,OTT0132,OtterBox Symmetry Alpha Glass Case + Screen Pr...,Pack OtterBox Symmetry Case + Screen Protector...,49.99,119.899,0,11865403
16628,APP2353,"Apple Macbook Pro 15 ""Core i7 Touch Bar 31GHz ...",New MacBook Pro 15-inch Core i7 Touch Bar 31Gh...,3999,3.719.004,0,"1,02E+12"
913,PAC0391,OWC Data Doubler Pack MacBook / Macbook Pro Black,Pack Replacement Superdrive for SSD / HDD + bo...,121.98,559.903,1,12755395
11418,PAC1258,Crucial SSD upgrade kit Kit tools MX200 1TB + ...,Crucial MX200 SSD + 1TB SSD installation kit i...,433.98,329.99,0,1433
11543,BEA0034,Beats by Dr. Dre Alone 2 Luxe Black Headphones,High-definition headphones with ultranitóda Re...,199.95,1.999.005,0,5384
17013,PAC1700,Pack QNAP TS-251A NAS Server | 16GB | 4TB (2x2...,NAS with 16GB of RAM and 4TB (2x2TB) WD Red fo...,709.67,641.179,0,12175397
10623,APP1216,Apple Magic Trackpad 2,Apple Wireless Bluetooth Trackpad.,149,139,1,1387


Both price and promo_price have some values with 2 '.' in them (e.g. 21.898.692). Probably, the first '.' should be the thousands separator, but can we be sure? (is 21,898 EUR a reasonable price for sku PAC1986, an NAS server with 16GB RAM and 25TB? Usual price for these is ~1700-2000 EUR)

promo_price also has strange values, e.g. (592.022, when price is 99.95)

Remove promo_price column as it is unreliable, and remove values from price with 2 '.'

Check how many rows in promo_price and price contain values with 2 '.'

In [ ]:
print(products['price'].str.count('\.').value_counts())
print(products['price'].str.count('\.').value_counts(normalize=True))

price
1.0    6942
0.0    3215
2.0     377
Name: count, dtype: int64
price
1.0    0.659009
0.0    0.305202
2.0    0.035789
Name: proportion, dtype: float64


<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-1110993471.py:1: SyntaxWarning: invalid escape sequence '\.'
  print(products['price'].str.count('\.').value_counts())
/tmp/ipython-input-1110993471.py:2: SyntaxWarning: invalid escape sequence '\.'
  print(products['price'].str.count('\.').value_counts(normalize=True))


377 (3.5%) of prices have 2 '.'

### Drop the rows with strange values of price (with 2 '.') as well as NAs


In [ ]:
products.shape

(10579, 7)

In [ ]:
#drop NAs from price
print(f'{len(products)} rows before dropping NAs from price')
products = products.dropna(subset = ['price'], axis = 0)
print(f'{len(products)} rows after dropping NAs from price')



10579 rows before dropping NAs from price
10534 rows after dropping NAs from price


In [ ]:
# drop rows with 2 '.' in price
print(f'{len(products)} rows before dropping rows with 2 "."s from price')
two_deci_count = (products["price"].str.count("\.")>1)
distorted_price_items = products.loc[two_deci_count, "sku"]
products = products.loc[~products["sku"].isin(distorted_price_items)]
print(f'{len(products)} rows after dropping rows with 2 "."s from price')

10534 rows before dropping rows with 2 "."s from price
10157 rows after dropping rows with 2 "."s from price


<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-411451528.py:3: SyntaxWarning: invalid escape sequence '\.'
  two_deci_count = (products["price"].str.count("\.")>1)


Make price numeric

In [ ]:
products['price'] = pd.to_numeric(products['price'])

# sense check for prices
products['price'].describe()


,price
count,10157.000000
mean,663.124629
std,1355.237789
min,2.990000
25%,44.900000
50%,119.000000
75%,649.891000
max,15339.000000


In [ ]:
maxprice = products['price'].max()
products[products['price']==maxprice]

,sku,name,desc,price,promo_price,in_stock,type
18429,APP2660,"Apple iMac Pro 27 ""18-core Intel Xeon W 23GHz ...",Pro iMac 27 inch screen Retina 5K and Intel Xe...,15339.0,144.190.049,0,118692158


15,339 EUR seems too expensive for this product (Google says it should be around 1,500 EUR)

Keep it in the analysis but **Note that some prices may not be correct**
(compare this price with orderlines price?)

**Recommend to review product price dataframe and catch invalid prices in data entry (e.g. prices with wrong format - 2 '.'s in price, or prices with 3 values after the decimal point)**

In [ ]:
#Read in the orderlines data to check the price against the products data for sku APP2660 (the expensive product above)
url = "https://drive.google.com/file/d/1FYhN_2AzTBFuWcfHaRuKcuCE6CWXsWtG/view?usp=drive_link" # orderlines.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines = pd.read_csv(path)

In [ ]:
orderlines[orderlines['sku'] == 'APP2660']

,id,id_order,product_id,product_quantity,sku,unit_price,date
232052,1551007,487164,0,1,APP2660,14.580.00,2018-01-09 23:33:16
232077,1551059,487192,0,1,APP2660,14.580.00,2018-01-10 00:05:59
232754,1551977,487490,0,1,APP2660,12.175.24,2018-01-10 10:15:49
234834,1555558,488935,0,4,APP2660,14.580.00,2018-01-11 21:11:34
241912,1567694,491691,0,1,APP2660,14.580.00,2018-01-18 14:40:05
248338,1577659,497375,0,1,APP2660,14.580.00,2018-01-23 23:55:41
253465,1585660,500546,0,1,APP2660,14.580.00,2018-01-29 01:55:02
255123,1587927,501275,0,1,APP2660,14.580.00,2018-01-29 17:18:55
255454,1588363,501413,0,1,APP2660,14.580.00,2018-01-29 18:58:03
282442,1632863,520029,0,1,APP2660,14.725.00,2018-03-02 11:57:52


The unit price for APP2660 in orderlines seems to be similar to the price in products, so keep it in.

Check how many promo_price rows have values with 2 '.'

In [ ]:
print(products['promo_price'].str.count('\.').value_counts())
print(products['promo_price'].str.count('\.').value_counts(normalize=True))

promo_price
1    5714
2    4321
0     122
Name: count, dtype: int64
promo_price
1    0.562568
2    0.425421
0    0.012011
Name: proportion, dtype: float64


<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-2518416103.py:1: SyntaxWarning: invalid escape sequence '\.'
  print(products['promo_price'].str.count('\.').value_counts())
/tmp/ipython-input-2518416103.py:2: SyntaxWarning: invalid escape sequence '\.'
  print(products['promo_price'].str.count('\.').value_counts(normalize=True))


4321 (42.5%) of rows have values with 2 '.'

Remove the whole column - too many corrupted values

In [ ]:
# Remove column with promo price

products = products.drop('promo_price', axis=1)
print(products.shape)

(10157, 6)


### Check NAs remaining

In [ ]:
products.isna().sum()

,0
sku,0
name,0
desc,6
price,0
in_stock,0
type,47


### Replace missing descriptions with name

In [ ]:
products.loc[products['desc'].isna(), 'desc'] = products.loc[products['desc'].isna(), 'name']

#alternative
#products['desc'] = products['desc'].fillna(products['name'])

In [ ]:
products[products['sku']=='WDT0211-A']

,sku,name,desc,price,in_stock,type
16126,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard ...","Open - Purple 2TB WD 35 ""PC Security Mac hard ...",107.0,0,1298


In [ ]:
products.isna().sum()

,0
sku,0
name,0
desc,0
price,0
in_stock,0
type,47


### Fill missing values in 'type' with 'None'

In [ ]:
products['type'] = products['type'].fillna('None')

### Save cleaned csv



In [ ]:
from google.colab import files

products.to_csv("products_clean.csv", index=False)
files.download("products_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>